Direct implementation as in paper


importing dataset and creating X(features) and y(label)

In [ ]:
import pandas as pd
df=pd.read_csv("data.csv")
df

,ID,air_time1,disp_index1,gmrt_in_air1,gmrt_on_paper1,max_x_extension1,max_y_extension1,mean_acc_in_air1,mean_acc_on_paper1,mean_gmrt1,...,mean_jerk_in_air25,mean_jerk_on_paper25,mean_speed_in_air25,mean_speed_on_paper25,num_of_pendown25,paper_time25,pressure_mean25,pressure_var25,total_time25,class
0,id_1,5160,0.000013,120.804174,86.853334,957,6601,0.361800,0.217459,103.828754,...,0.141434,0.024471,5.596487,3.184589,71,40120,1749.278166,296102.7676,144605,P
1,id_2,51980,0.000016,115.318238,83.448681,1694,6998,0.272513,0.144880,99.383459,...,0.049663,0.018368,1.665973,0.950249,129,126700,1504.768272,278744.2850,298640,P
2,id_3,2600,0.000010,229.933997,172.761858,2333,5802,0.387020,0.181342,201.347928,...,0.178194,0.017174,4.000781,2.392521,74,45480,1431.443492,144411.7055,79025,P
3,id_4,2130,0.000010,369.403342,183.193104,1756,8159,0.556879,0.164502,276.298223,...,0.113905,0.019860,4.206746,1.613522,123,67945,1465.843329,230184.7154,181220,P
4,id_5,2310,0.000007,257.997131,111.275889,987,4732,0.266077,0.145104,184.636510,...,0.121782,0.020872,3.319036,1.680629,92,37285,1841.702561,158290.0255,72575,P
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,id_170,2930,0.000010,241.736477,176.115957,1839,6439,0.253347,0.174663,208.926217,...,0.119152,0.020909,4.508709,2.233198,96,44545,1798.923336,247448.3108,80335,H
170,id_171,2140,0.000009,274.728964,234.495802,2053,8487,0.225537,0.174920,254.612383,...,0.174495,0.017640,4.685573,2.806888,84,37560,1725.619941,160664.6464,345835,H
171,id_172,3830,0.000008,151.536989,171.104693,1287,7352,0.165480,0.161058,161.320841,...,0.114472,0.017194,3.493815,2.510601,88,51675,1915.573488,128727.1241,83445,H
172,id_173,1760,0.000008,289.518195,196.411138,1674,6946,0.518937,0.202613,242.964666,...,0.114472,0.017194,3.493815,2.510601,88,51675,1915.573488,128727.1241,83445,H


In [ ]:
import numpy as np
features_column=df.columns[1:-1] #except id and label
label_column=df.columns[-1] #only label
X=df[features_column].values #returns only the values of the column
y_text=df[label_column].values
y=(y_text=='P').astype(int) #1-P and 0-H

**classical SVC**

PCA=24, split=60/20/20(training/validation/test)

In [ ]:
from sklearn.model_selection import GridSearchCV,ShuffleSplit,train_test_split #for hyperparameter search
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
import numpy as np
outer_cv=ShuffleSplit(n_splits=20,test_size=0.2,random_state=19)
pipe_svc=Pipeline([('scaler1',StandardScaler()),('pca',PCA(n_components=24,random_state=42)),('scaler2',StandardScaler()),('svc',SVC(random_state=42))])
param_grid_svc={'svc__kernel':['linear','poly','rbf','sigmoid'],
                'svc__C':[0.1,1,10,100],
                'svc__gamma':['scale',1,0.1,0.01,0.001],
                'svc__tol':[1e-3,1e-4,1e-5]}
acc_svc_tuned=[]
for i,(trainval_idx,test_idx) in enumerate(outer_cv.split(X,y)):
    X_trainval,X_test=X[trainval_idx],X[test_idx]
    y_trainval,y_test=y[trainval_idx],y[test_idx]
    X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.25,stratify=y_trainval,random_state=42)# further splitting
    grid_svc=GridSearchCV( estimator=pipe_svc, #model to tune
                          param_grid=param_grid_svc, # parameters to try
                          cv=3, #no of folds
                          scoring='accuracy', #how to measure the performace
                          n_jobs=-1 #all cpu cores to speed the training
                         ) #grid search on only train vs val
    grid_svc.fit(X_train,y_train)
    best_model=grid_svc.best_estimator_
    best_model.fit(np.vstack([X_train,X_val]),np.concatenate([y_train,y_val])) #joins two part to form train+val set
    acc=best_model.score(X_test,y_test)
    acc_svc_tuned.append(acc)
    print(f"split{i+1}:{acc:.2f}")
mean_acc_svc=np.mean(acc_svc_tuned)*100
std_acc_svc=np.std(acc_svc_tuned)*100
print(f"\n mean accuracy:{mean_acc_svc:.2f}")
print(f"\n std accuracy:{std_acc_svc:.2f}")





split1:0.80
split2:0.83
split3:0.71
split4:0.83
split5:0.86
split6:0.80
split7:0.89
split8:0.83
split9:0.80
split10:0.80
split11:0.83
split12:0.91
split13:0.86
split14:0.74
split15:0.91
split16:0.89
split17:0.91
split18:0.91
split19:0.89
split20:0.77

 mean accuracy:83.86

 std accuracy:5.73


**Classical KNN**




In [ ]:
from sklearn.model_selection import GridSearchCV,ShuffleSplit,train_test_split #for hyperparameter search
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
outer_cv=ShuffleSplit(n_splits=20,test_size=0.2,random_state=19)
pipe_knn=Pipeline([('scaler1',StandardScaler()),('pca',PCA(n_components=24,random_state=42)),('scaler2',StandardScaler()),('knn',KNeighborsClassifier())])
param_grid_knn={
    'knn__n_neighbors':[3,5,7,9],
    'knn__metric':['euclidean','manhattan','minkowski'],
    'knn__weights':['uniform','distance']}
acc_knn_tuned=[]
for i,(trainval_idx,test_idx) in enumerate(outer_cv.split(X,y)):
    X_trainval,X_test=X[trainval_idx],X[test_idx]
    y_trainval,y_test=y[trainval_idx],y[test_idx]
    X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.25,stratify=y_trainval,random_state=42)# further splitting
    grid_knn=GridSearchCV( estimator=pipe_knn, #model to tune
                          param_grid=param_grid_knn, # parameters to try
                          cv=3, #no of folds
                          scoring='accuracy', #how to measure the performace
                          n_jobs=-1 #all cpu cores to speed the training
                         ) #grid search on only train vs val
    grid_knn.fit(X_train,y_train)
    best_model=grid_knn.best_estimator_
    best_model.fit(np.vstack([X_train,X_val]),np.concatenate([y_train,y_val])) #joins two part to form train+val set
    acc=best_model.score(X_test,y_test)
    acc_knn_tuned.append(acc)
    print(f"split{i+1}:{acc:.2f}")
mean_acc_knn=np.mean(acc_knn_tuned)*100
std_acc_knn=np.std(acc_knn_tuned)*100
print(f"\n mean accuracy:{mean_acc_knn:.2f}")
print(f"\n std accuracy:{std_acc_knn:.2f}")





split1:0.54
split2:0.60
split3:0.60
split4:0.71
split5:0.66
split6:0.83
split7:0.69
split8:0.69
split9:0.57
split10:0.74
split11:0.77
split12:0.77
split13:0.71
split14:0.74
split15:0.69
split16:0.86
split17:0.86
split18:0.66
split19:0.66
split20:0.46

 mean accuracy:69.00

 std accuracy:10.11


**classical DT**

In [ ]:
from sklearn.model_selection import GridSearchCV,ShuffleSplit,train_test_split #for hyperparameter search
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
import numpy as np
outer_cv=ShuffleSplit(n_splits=20,test_size=0.2,random_state=42)
pipe_dt=Pipeline([('scaler1',StandardScaler()),('pca',PCA(n_components=24,random_state=42)),('scaler2',StandardScaler()),('dt',DecisionTreeClassifier(random_state=42))])
param_grid_dt={
    'dt__criterion':['gini','entropy'],
    'dt__splitter':['best','random'],
    'dt__max_depth':[None,10,20,30],
    'dt__min_samples_split':[2,5,10],
    'dt__min_samples_leaf':[1,2,4]
}
acc_dt_tuned=[]
for i,(trainval_idx,test_idx) in enumerate(outer_cv.split(X,y)):
    X_trainval,X_test=X[trainval_idx],X[test_idx]
    y_trainval,y_test=y[trainval_idx],y[test_idx]
    X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.25,stratify=y_trainval,random_state=42)# further splitting
    grid_dt=GridSearchCV( estimator=pipe_dt, #model to tune
                          param_grid=param_grid_dt, # parameters to try
                          cv=3, #no of folds
                          scoring='accuracy', #how to measure the performace
                          n_jobs=-1 #all cpu cores to speed the training
                         ) #grid search on only train vs val
    grid_dt.fit(X_train,y_train)
    best_model=grid_dt.best_estimator_
    best_model.fit(np.vstack([X_train,X_val]),np.concatenate([y_train,y_val])) #joins two part to form train+val set
    acc=best_model.score(X_test,y_test)
    acc_dt_tuned.append(acc)
    print(f"split{i+1}:{acc:.2f}")
mean_acc_dt=np.mean(acc_dt_tuned)*100
std_acc_dt=np.std(acc_dt_tuned)*100
print(f"\n mean accuracy:{mean_acc_dt:.2f}")
print(f"\n std accuracy:{std_acc_dt:.2f}")





split1:0.89
split2:0.74
split3:0.69
split4:0.80
split5:0.74
split6:0.66
split7:0.66
split8:0.71
split9:0.66
split10:0.83
split11:0.80
split12:0.71
split13:0.66
split14:0.74
split15:0.66
split16:0.66
split17:0.74
split18:0.69
split19:0.83
split20:0.69

 mean accuracy:72.71

 std accuracy:6.79


***Final Table***

In [ ]:
print("Classical\n")
print("SVC","KNN","DT",sep="\t\t")
print(
    f"{mean_acc_svc:.2f}±{std_acc_svc:.2f}",
    f"{mean_acc_knn:.2f}±{std_acc_knn:.2f}",
    f"{mean_acc_dt:.2f}±{std_acc_dt:.2f}",
    sep="\t"
)


Classical

SVC		KNN		DT
83.86±5.73	69.00±10.11	72.71±6.79


**Conclusion : SVC provides better efficiency than KTT and DT**